In [32]:
help(dm.descriptors.batch_compute_many_descriptors)

Help on function batch_compute_many_descriptors in module datamol.descriptors.compute:

batch_compute_many_descriptors(
    mols: List[Mol],
    properties_fn: Optional[Dict[str, Union[Callable, str]]] = None,
    add_properties: bool = True,
    n_jobs: int = 1,
    batch_size: Optional[int] = None,
    progress: bool = False,
    progress_leave: bool = True
) -> pd.DataFrame
    Compute a list of opiniated molecular properties on a list of molecules.

    Args:
        mols: A list of molecules.
        properties_fn: A list of functions that compute properties. If None,
            a default list of properties is used. If the function is a string,
            `dm.descriptors.any_descriptor()` is used to retrieve the descriptor
            function.
        add_properties: Whether to add the computed properties to the default list.

    Returns:
        A dataframe of computed properties with one row per input molecules.



In [3]:
import pandas as pd
import datamol as dm
from tqdm.auto import tqdm
from rdkit.Chem import rdDepictor
import mols2grid

In [4]:
rdDepictor.SetPreferCoordGen(True)
pd.options.display.float_format = '{:,.2f}'.format

In [5]:
url = "https://raw.githubusercontent.com/PatWalters/yamc/main/data/HERG.smi"
df = pd.read_csv(url,sep=" ",names=["SMILES","Name","pIC50"])

In [6]:
df.head()

,SMILES,Name,pIC50
0,c1cc(ccc1n2cc(c3c2ccc(c3)Cl)C4CCN(CC4)CCN5CCNC...,CHEMBL12713,8.03
1,c1ccc2c(c1)[nH]c(=O)n2C3=CCN(CC3)CCCC(=O)c4ccc...,CHEMBL1108,7.33
2,c1ccc2c(c1)c(c[nH]2)C(=O)OC3CC4CC5CC(C3)N4CC5=O,CHEMBL2368925,5.07
3,CC(C)C(CCCN(C)CCc1ccc(c(c1)OC)OC)(C#N)c2ccc(c(...,CHEMBL6966,6.83
4,CCOC(=O)N1CCC(=C2c3ccc(cc3CCc4c2nccc4)Cl)CC1,CHEMBL998,6.43


In [7]:
df['IC50'] = dm.molar.log_to_molar(df.pIC50,'uM')
df.head()

,SMILES,Name,pIC50,IC50
0,c1cc(ccc1n2cc(c3c2ccc(c3)Cl)C4CCN(CC4)CCN5CCNC...,CHEMBL12713,8.03,0.01
1,c1ccc2c(c1)[nH]c(=O)n2C3=CCN(CC3)CCCC(=O)c4ccc...,CHEMBL1108,7.33,0.05
2,c1ccc2c(c1)c(c[nH]2)C(=O)OC3CC4CC5CC(C3)N4CC5=O,CHEMBL2368925,5.07,8.48
3,CC(C)C(CCCN(C)CCc1ccc(c(c1)OC)OC)(C#N)c2ccc(c(...,CHEMBL6966,6.83,0.15
4,CCOC(=O)N1CCC(=C2c3ccc(cc3CCc4c2nccc4)Cl)CC1,CHEMBL998,6.43,0.37


In [8]:
df['mol'] = dm.from_df(df, smiles_column="SMILES")
df.head()

,SMILES,Name,pIC50,IC50,mol
0,c1cc(ccc1n2cc(c3c2ccc(c3)Cl)C4CCN(CC4)CCN5CCNC...,CHEMBL12713,8.03,0.01,<rdkit.Chem.rdchem.Mol object at 0x30c31b5a0>
1,c1ccc2c(c1)[nH]c(=O)n2C3=CCN(CC3)CCCC(=O)c4ccc...,CHEMBL1108,7.33,0.05,<rdkit.Chem.rdchem.Mol object at 0x30c31b610>
2,c1ccc2c(c1)c(c[nH]2)C(=O)OC3CC4CC5CC(C3)N4CC5=O,CHEMBL2368925,5.07,8.48,<rdkit.Chem.rdchem.Mol object at 0x30c31b680>
3,CC(C)C(CCCN(C)CCc1ccc(c(c1)OC)OC)(C#N)c2ccc(c(...,CHEMBL6966,6.83,0.15,<rdkit.Chem.rdchem.Mol object at 0x30c31b6f0>
4,CCOC(=O)N1CCC(=C2c3ccc(cc3CCc4c2nccc4)Cl)CC1,CHEMBL998,6.43,0.37,<rdkit.Chem.rdchem.Mol object at 0x30c31b760>


In [9]:
def max_ring_size(mol):
    """Get the size of the largest ring in a molecule

    :param mol: input_molecule
    :return: size of the largest ring or 0 for an acyclic molecule
    """
    ri = mol.GetRingInfo()
    atom_rings = ri.AtomRings()
    if len(atom_rings) == 0:
        return 0
    else:
        return max([len(x) for x in ri.AtomRings()])

In [10]:
my_prop_dict = {
    "mw" : dm.descriptors.mw,
    "logp" : dm.descriptors.clogp,
    "hbd" : dm.descriptors.n_lipinski_hbd,
    "hba" : dm.descriptors.n_lipinski_hba,
    "max_ring_size" : max_ring_size
}

In [11]:
prop_df = dm.descriptors.batch_compute_many_descriptors(df.mol,properties_fn=my_prop_dict,add_properties=False,
                                             progress=True)

  0%|          | 0/4042 [00:00<?, ?it/s]

In [12]:
prop_df

,mw,logp,hbd,hba,max_ring_size
0,440.18,4.63,1,5,6
1,379.17,3.68,1,5,6
2,324.15,2.52,1,5,6
3,454.28,5.09,0,6,6
4,382.14,4.89,0,4,7
...,...,...,...,...,...
4037,493.21,3.38,2,10,6
4038,480.18,3.16,2,10,6
4039,494.19,3.81,1,10,6
4040,494.19,3.55,2,10,6


In [13]:
df = pd.concat([df,prop_df],axis=1)

In [14]:
df.head()

,SMILES,Name,pIC50,IC50,mol,mw,logp,hbd,hba,max_ring_size
0,c1cc(ccc1n2cc(c3c2ccc(c3)Cl)C4CCN(CC4)CCN5CCNC...,CHEMBL12713,8.03,0.01,<rdkit.Chem.rdchem.Mol object at 0x30c31b5a0>,440.18,4.63,1,5,6
1,c1ccc2c(c1)[nH]c(=O)n2C3=CCN(CC3)CCCC(=O)c4ccc...,CHEMBL1108,7.33,0.05,<rdkit.Chem.rdchem.Mol object at 0x30c31b610>,379.17,3.68,1,5,6
2,c1ccc2c(c1)c(c[nH]2)C(=O)OC3CC4CC5CC(C3)N4CC5=O,CHEMBL2368925,5.07,8.48,<rdkit.Chem.rdchem.Mol object at 0x30c31b680>,324.15,2.52,1,5,6
3,CC(C)C(CCCN(C)CCc1ccc(c(c1)OC)OC)(C#N)c2ccc(c(...,CHEMBL6966,6.83,0.15,<rdkit.Chem.rdchem.Mol object at 0x30c31b6f0>,454.28,5.09,0,6,6
4,CCOC(=O)N1CCC(=C2c3ccc(cc3CCc4c2nccc4)Cl)CC1,CHEMBL998,6.43,0.37,<rdkit.Chem.rdchem.Mol object at 0x30c31b760>,382.14,4.89,0,4,7


In [15]:
df = df[df['mw'] <= 500]
df = df[df['logp'] <= 5]
df = df[df['hbd'] <= 5]
df = df[df['hba'] <= 10]

In [16]:
df_ro5_ok = df.query("mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10")
len(df_ro5_ok)

3477

In [17]:
df['ro5_ok'] = (df.mw <= 500) & (df.logp <=5) & (df.hbd <= 5) & (df.hba <= 10)
df.ro5_ok.sum()

np.int64(3477)

In [18]:
def bool_to_int(bool_in):
    return bool_in.values.astype(int)

In [19]:
df['ro5_count'] = bool_to_int(df.mw <= 500) + bool_to_int(df.logp <=5) + bool_to_int(df.hbd <= 5) + bool_to_int(df.hba <= 10)
len(df.query("ro5_count == 4"))

3477

In [23]:
df = df.reset_index(drop=True)
# cluster the molecules
cluster_list = dm.cluster_mols(df.mol)
# create a list to hold the cluster ids, which we will add to the dataframe
cluster_idx = [-1] * len(df)
for i,cluster in enumerate(tqdm(cluster_list[0])):
    # align the structures for each cluster using Bemis Murcko frameworks
    dm.align.auto_align_many([df.mol.values[x] for x in cluster],copy=False,partition_method='scaffold')
    # add the cluster id to cluster_idx
    for c in cluster:
        cluster_idx[c] = i
# add a column with cluster ids to the dataframe
df['cluster'] = cluster_idx

  0%|          | 0/3026 [00:00<?, ?it/s]

In [24]:
cluster_sample_df = df.sort_values("cluster").drop_duplicates("cluster").copy()
cluster_sample_df.head()

,SMILES,Name,pIC50,IC50,mol,mw,logp,hbd,hba,max_ring_size,ro5_ok,ro5_count,cluster
972,CC(C)Oc1ccc(cc1)C2CN(C(=O)N2CCc3ccc(cc3)OC)NS(...,CHEMBL571952,4.77,16.98,<rdkit.Chem.rdchem.Mol object at 0x30c36e500>,447.18,2.97,1,8,6,True,4,0
2054,c1ccc(c(c1)C(=O)Nc2ccc(cc2)c3nnc(o3)NCCCCN4CCC...,CHEMBL2022733,4.90,12.59,<rdkit.Chem.rdchem.Mol object at 0x30e40c970>,437.22,4.81,2,7,6,True,4,1
1066,Cc1c(ocn1)c2nnc(n2C)SCCCN3CC4CC4(C3)c5ccc(cc5)...,CHEMBL1079823,6.40,0.40,<rdkit.Chem.rdchem.Mol object at 0x30c3753f0>,451.24,4.83,0,6,6,True,4,2
1061,Cc1c(ocn1)c2nnc(n2C)SCCCN3CC4CC4(C3)c5ccc(cc5)Cl,CHEMBL1080490,6.39,0.41,<rdkit.Chem.rdchem.Mol object at 0x30c375150>,429.14,4.19,0,6,6,True,4,3
3358,COc1ccc(cc1)c2nnc(o2)C(=O)N3CCC(CC3)Oc4ccc(cc4...,CHEMBL3786572,4.96,10.96,<rdkit.Chem.rdchem.Mol object at 0x30e443760>,462.23,4.02,0,8,6,True,4,4


In [25]:
cluster_count_df = df.cluster.value_counts().to_frame().reset_index()
cluster_count_df.columns = ["cluster","count"]
cluster_sample_df = cluster_sample_df.merge(cluster_count_df,on="cluster")
cluster_sample_df.head()

,SMILES,Name,pIC50,IC50,mol,mw,logp,hbd,hba,max_ring_size,ro5_ok,ro5_count,cluster,count
0,CC(C)Oc1ccc(cc1)C2CN(C(=O)N2CCc3ccc(cc3)OC)NS(...,CHEMBL571952,4.77,16.98,<rdkit.Chem.rdchem.Mol object at 0x30c36e500>,447.18,2.97,1,8,6,True,4,0,10
1,c1ccc(c(c1)C(=O)Nc2ccc(cc2)c3nnc(o3)NCCCCN4CCC...,CHEMBL2022733,4.90,12.59,<rdkit.Chem.rdchem.Mol object at 0x30e40c970>,437.22,4.81,2,7,6,True,4,1,9
2,Cc1c(ocn1)c2nnc(n2C)SCCCN3CC4CC4(C3)c5ccc(cc5)...,CHEMBL1079823,6.40,0.40,<rdkit.Chem.rdchem.Mol object at 0x30c3753f0>,451.24,4.83,0,6,6,True,4,2,8
3,Cc1c(ocn1)c2nnc(n2C)SCCCN3CC4CC4(C3)c5ccc(cc5)Cl,CHEMBL1080490,6.39,0.41,<rdkit.Chem.rdchem.Mol object at 0x30c375150>,429.14,4.19,0,6,6,True,4,3,4
4,COc1ccc(cc1)c2nnc(o2)C(=O)N3CCC(CC3)Oc4ccc(cc4...,CHEMBL3786572,4.96,10.96,<rdkit.Chem.rdchem.Mol object at 0x30e443760>,462.23,4.02,0,8,6,True,4,4,7


In [26]:
mols2grid.display(cluster_sample_df,mol_col="mol",subset=["img","cluster","count"])

In [27]:
def show_cluster(x):
    return mols2grid.display(df.query(f"cluster == {x}"),mol_col="mol",subset=["img","Name","pIC50"],use_coords=True, prerender=True, transform = {"pIC50" : lambda val: f"{val:.2f}"})

In [28]:
show_cluster(1)

In [33]:
cluster_list = dm.cluster_mols(df.mol)

In [34]:
cluster_list

(((971, 966, 967, 968, 969, 970, 972, 973, 976, 977),
  (2022, 2023, 2024, 2025, 2044, 2046, 2047, 2052, 2054),
  (1063, 1044, 1051, 1052, 1059, 1064, 1066, 1068),
  (1061, 1042, 1048, 1060),
  (3358, 3356, 3357, 3359, 3360, 3363, 3364),
  (2827, 2820, 2821, 2822, 2828, 2829),
  (2050, 2051, 2060),
  (442, 414, 418, 438, 439, 440),
  (3467, 3473, 3474, 3475, 3476),
  (3347, 2178, 2179, 3341, 3346),
  (3331, 3122, 3123, 3332, 3333),
  (3321, 3312, 3315, 3318, 3323),
  (2937, 2933, 2934, 2935, 2936),
  (2868, 2865, 2869, 2871, 2873),
  (2493, 2491, 2494, 2495, 2496),
  (1799, 1796, 1797, 1798, 1800),
  (3280, 3281, 3282, 3283),
  (3197, 3175, 3189, 3195),
  (3152, 3144, 3147, 3151),
  (3067, 2927, 3064, 3066),
  (2872, 2861, 2862, 2863),
  (2567, 2560, 2565, 2566),
  (2056, 2032, 2033, 2034),
  (2039, 2040, 2041, 2057),
  (1939, 1936, 1937, 1938),
  (1680, 1151, 1155, 1676),
  (1633, 768, 769, 782),
  (1582, 1432, 1580, 1581),
  (1563, 1556, 1557, 1562),
  (1561, 1554, 1555, 1560),
  (10

In [36]:
len(df)


3477